# Phase 4 — Premier modèle de détection des canulars

## Objectifs

- Recharger et préparer les relevés structurés.
- Reconstruire la cible artificielle `is_hoax`.
- Séparer les données entre entraînement et test.
- Entraîner un premier modèle de classification.
- Évaluer le modèle sur des données jamais vues pendant l'entraînement.
- Mesurer la precision et le recall.

> Attention : ce premier modèle utilise volontairement la colonne `comments`.
> La phase 5 vérifiera si cette information constitue une fuite de données.

## Imports

In [1]:
from pathlib import Path
import csv
import re

import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    precision_score,
    recall_score,
)
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline

## Chemin et Noms de colonne

In [2]:
DATA_PATH = Path("../data/releves_klaxo3.csv")

OUTPUT_DIR = Path("../outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

COLUMNS = [
    "datetime",
    "city",
    "state",
    "country",
    "shape",
    "duration_seconds",
    "duration_hours_min",
    "comments",
    "date_posted",
    "latitude",
    "longitude",
]

## Charger les lignes valides

In [3]:
lignes_valides = []
lignes_problemes = []

with open(DATA_PATH, "r", encoding="utf-8", errors="replace", newline="") as f:
    reader = csv.reader(f)

    for numero_ligne, row in enumerate(reader, start=1):
        if len(row) == len(COLUMNS):
            lignes_valides.append(row)
        else:
            lignes_problemes.append({
                "numero_ligne": numero_ligne,
                "nb_champs": len(row),
                "contenu": row,
            })

df = pd.DataFrame(lignes_valides, columns=COLUMNS)

print(f"Lignes exploitables chargées : {len(df)}")
print(f"Lignes isolées : {len(lignes_problemes)}")

Lignes exploitables chargées : 88679
Lignes isolées : 196


## Convertir les types

In [4]:
colonnes_numeriques = [
    "duration_seconds",
    "latitude",
    "longitude",
]

colonnes_dates = [
    "datetime",
    "date_posted",
]

for col in colonnes_numeriques:
    df[col] = pd.to_numeric(df[col], errors="coerce")

for col in colonnes_dates:
    df[col] = pd.to_datetime(df[col], errors="coerce")

df.dtypes

datetime              datetime64[ns]
city                          object
state                         object
country                       object
shape                         object
duration_seconds             float64
duration_hours_min            object
comments                      object
date_posted           datetime64[ns]
latitude                     float64
longitude                    float64
dtype: object

## Recréer la cible `is_hoax`

In [5]:
MOTS_CLES_CANULAR = [
    "hoax",
    "fake",
    "prank",
    "joke",
    "not real",
    "made up",
    "fraud",
]

pattern_canular = "|".join(
    re.escape(mot) for mot in MOTS_CLES_CANULAR
)

df["comments_clean"] = (
    df["comments"]
    .fillna("")
    .astype(str)
    .str.lower()
)

df["is_hoax"] = (
    df["comments_clean"]
    .str.contains(
        pattern_canular,
        regex=True,
        na=False,
    )
    .astype(int)
)

print(df["is_hoax"].value_counts())
print(df["is_hoax"].value_counts(normalize=True))

is_hoax
0    87810
1      869
Name: count, dtype: int64
is_hoax
0    0.990201
1    0.009799
Name: proportion, dtype: float64


## Créer des variables à partir des dates

In [6]:
df["observation_year"] = df["datetime"].dt.year
df["observation_month"] = df["datetime"].dt.month
df["observation_hour"] = df["datetime"].dt.hour

df["posted_year"] = df["date_posted"].dt.year

df[
    [
        "datetime",
        "observation_year",
        "observation_month",
        "observation_hour",
        "date_posted",
        "posted_year",
    ]
].head()

,datetime,observation_year,observation_month,observation_hour,date_posted,posted_year
0,1949-10-10 20:30:00,1949.0,10.0,20.0,2004-04-27,2004
1,1949-10-10 21:00:00,1949.0,10.0,21.0,2005-12-16,2005
2,1955-10-10 17:00:00,1955.0,10.0,17.0,2008-01-21,2008
3,1956-10-10 21:00:00,1956.0,10.0,21.0,2004-01-17,2004
4,1960-10-10 20:00:00,1960.0,10.0,20.0,2004-01-22,2004


## Créer une colonne de textes communes

In [7]:
df["text_features_with_leakage"] = (
    "comment " + df["comments"].fillna("").astype(str)
    + " city " + df["city"].fillna("").astype(str)
    + " state " + df["state"].fillna("").astype(str)
    + " country " + df["country"].fillna("").astype(str)
    + " shape " + df["shape"].fillna("").astype(str)
)

df[
    [
        "text_features_with_leakage",
        "is_hoax"
    ]
].head()

,text_features_with_leakage,is_hoax
0,comment This event took place in early fall ar...,0
1,comment 1949 Lackland AFB&#44 TX. Lights raci...,0
2,comment Green/Orange circular disc over Cheste...,0
3,comment My older brother and twin sister were ...,0
4,comment AS a Marine 1st Lt. flying an FJ4B fig...,0


## Créer les entrées `x` et la cible `y`

In [8]:
features_numeriques = [
    "duration_seconds",
    "latitude",
    "longitude",
    "observation_year",
    "observation_month",
    "observation_hour",
    "posted_year",
]

features_modele = [
    "text_features_with_leakage",
] + features_numeriques

X = df[features_modele].copy()
y = df["is_hoax"].copy()

print("Dimensions de X :", X.shape)
print("Dimensions de y :", y.shape)

Dimensions de X : (88679, 8)
Dimensions de y : (88679,)


## Séparer train et test

In [9]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y,
)

print("Taille entraînement :", X_train.shape)
print("Taille test :", X_test.shape)

print(f"Part de canulars dans train : {y_train.mean():.2%}")
print(f"Part de canulars dans test : {y_test.mean():.2%}")

Taille entraînement : (70943, 8)
Taille test : (17736, 8)
Part de canulars dans train : 0.98%
Part de canulars dans test : 0.98%


## Construire le prétraitement

In [10]:
preprocessing = ColumnTransformer(
    transformers=[
        (
            "texte",
            TfidfVectorizer(
                lowercase=True,
                min_df=2,
                max_features=30_000,
                ngram_range=(1, 2),
            ),
            "text_features_with_leakage",
        ),
        (
            "numerique",
            Pipeline(
                steps=[
                    ("imputer", SimpleImputer(strategy="median")),
                ]
            ),
            features_numeriques,
        ),
    ]
)

preprocessing

,transformers,"[('texte', ...), ('numerique', ...)]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True
,force_int_remainder_cols,'deprecated'
,input,'content'
,encoding,'utf-8'
,decode_error,'strict'


## Construire le modèle complet

In [11]:
modele_initial = Pipeline(
    steps=[
        ("preprocessing", preprocessing),
        (
            "classifier",
            LogisticRegression(
                max_iter=1000,
                class_weight="balanced",
                random_state=42,
            ),
        ),
    ]
)

modele_initial

,steps,"[('preprocessing', ...), ('classifier', ...)]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('texte', ...), ('numerique', ...)]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


## Entraîner le model

In [12]:
modele_initial.fit(X_train, y_train)

print("Entraînement terminé.")

Entraînement terminé.


c:\Users\serge\anaconda3\Lib\site-packages\sklearn\linear_model\_logistic.py:473: ConvergenceWarning: lbfgs failed to converge after 1000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=1000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


## Prédire sur le jeu de test

In [13]:
y_pred_initial = modele_initial.predict(X_test)

pd.Series(y_pred_initial).value_counts()

0    10958
1     6778
Name: count, dtype: int64

## Calculer les métriques exigées

In [14]:
precision_initiale = precision_score(
    y_test,
    y_pred_initial,
    zero_division=0,
)

recall_initial = recall_score(
    y_test,
    y_pred_initial,
    zero_division=0,
)

accuracy_initiale = accuracy_score(
    y_test,
    y_pred_initial,
)

print(f"Precision : {precision_initiale:.2%}")
print(f"Recall : {recall_initial:.2%}")
print(f"Accuracy : {accuracy_initiale:.2%}")

Precision : 1.65%
Recall : 64.37%
Accuracy : 62.07%


## Matrice de confusion

In [15]:
matrice = confusion_matrix(y_test, y_pred_initial)

df_matrice = pd.DataFrame(
    matrice,
    index=[
        "Réel : non-canular",
        "Réel : canular",
    ],
    columns=[
        "Prédit : non-canular",
        "Prédit : canular",
    ],
)

df_matrice

,Prédit : non-canular,Prédit : canular
Réel : non-canular,10896,6666
Réel : canular,62,112


## Rapport de classification détaillé

In [16]:
print(
    classification_report(
        y_test,
        y_pred_initial,
        target_names=[
            "non-canular",
            "canular",
        ],
        zero_division=0,
    )
)

              precision    recall  f1-score   support

 non-canular       0.99      0.62      0.76     17562
     canular       0.02      0.64      0.03       174

    accuracy                           0.62     17736
   macro avg       0.51      0.63      0.40     17736
weighted avg       0.98      0.62      0.76     17736



## Enregistrer les résultats

In [17]:
resultats_modele_initial = pd.DataFrame(
    [
        {
            "modele": "Premier modèle avec commentaires",
            "precision": precision_initiale,
            "recall": recall_initial,
            "accuracy": accuracy_initiale,
            "taille_train": len(X_train),
            "taille_test": len(X_test),
            "canulars_train": int(y_train.sum()),
            "canulars_test": int(y_test.sum()),
        }
    ]
)

resultats_modele_initial.to_csv(
    OUTPUT_DIR / "resultats_modele_initial.csv",
    index=False,
)

df_matrice.to_csv(
    OUTPUT_DIR / "matrice_confusion_modele_initial.csv",
    index=True,
)

resultats_modele_initial

,modele,precision,recall,accuracy,taille_train,taille_test,canulars_train,canulars_test
0,Premier modèle avec commentaires,0.016524,0.643678,0.620659,70943,17736,695,174
